<a href="https://colab.research.google.com/github/SATHRAMCHARAN/CSA6102---DIGITAL-FORENSICS-AND-CYBERCRIME-INVENTIGATION/blob/main/exp32.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
from datetime import datetime

TS_FMT = "%Y-%m-%d %H:%M:%S"

def detect_timestomping(file_meta, change_gap_minutes=60):
    """
    file_meta: {
        "modified",
        "accessed",
        "changed",
        "born"
    } as timestamp strings.

    Returns:
        (is_suspicious: bool, reasons: list[str])
    """

    m = datetime.strptime(file_meta["modified"], TS_FMT)
    a = datetime.strptime(file_meta["accessed"], TS_FMT)
    c = datetime.strptime(file_meta["changed"], TS_FMT)
    b = datetime.strptime(file_meta["born"], TS_FMT)

    reasons = []

    if m < b:
        reasons.append("Modified time is earlier than Born (creation) time")

    if a < b:
        reasons.append("Accessed time is earlier than Born (creation) time")

    gap_minutes = abs((c - m).total_seconds()) / 60

    if gap_minutes > change_gap_minutes and c > m:
        reasons.append(
            f"MFT Changed time is {gap_minutes:.0f} minutes after Modified time "
            "— metadata may have been altered after the fact"
        )

    return (len(reasons) > 0, reasons)


# Example usage
file_meta = {
    "modified": "2026-08-06 09:00:00",
    "accessed": "2026-08-06 09:10:00",
    "changed": "2026-08-06 11:30:00",
    "born": "2026-08-06 10:00:00"
}

suspicious, reasons = detect_timestomping(file_meta)

print("Suspicious:", suspicious)
print("Reasons:")
for reason in reasons:
    print("-", reason)

Suspicious: True
Reasons:
- Modified time is earlier than Born (creation) time
- Accessed time is earlier than Born (creation) time
- MFT Changed time is 150 minutes after Modified time — metadata may have been altered after the fact
